# Ordered Logistic Regression Results: Adoption Predictors in Rangeland Management (FAIR^2) Exploration with `mlcroissant`

This notebook provides a comprehensive walkthrough for loading, exploring, and processing the "Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya" dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant JSON-LD schema and is accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and discover available record sets and fields using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL for FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata and define a dataset variable
dataset = mlc.Dataset(croissant_url)

# Access metadata (as object, not dict)
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets and their field `@id`s.

> **Note:** All entities are referenced by their `@id` fields for clarity and reproducibility.

In [ ]:
# List available record sets
record_sets = list(dataset.record_sets)
print("Available Record Sets (@id and name):\n")
for rs in record_sets:
    print(f"- {rs.id}: {getattr(rs, 'name', '[no name]')}")

# Show fields and their @id for each record set
record_set_ids = [rs.id for rs in record_sets]
record_set_fields = {}
for rs in record_sets:
    print(f"\nFields in Record Set '{rs.id}':")
    for field in rs.fields:
        print(f"  - Field @id: {field.id} (name: {getattr(field, 'name', '')}, dataType: {getattr(field, 'data_type', '')})")
    record_set_fields[rs.id] = [f.id for f in rs.fields]

## 3. Data Extraction
Extract and preview data from a selected record set into a DataFrame. All operations use `@id` for referencing the record set and fields.

In [ ]:
# Example: Load all available record sets into DataFrames by @id
dfs = {}

for rs_id in record_set_ids:
    # Each dataset.records yields dicts with keys as field @id
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        dfs[rs_id] = pd.DataFrame(records)

# Show columns (@id) of each DataFrame
for rs_id, df in dfs.items():
    print(f"\nDataFrame for Record Set '@id': {rs_id}")
    print("Columns (field @id):", df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
We'll select a record set with numeric data for demonstration. We'll:
- Filter records by a numeric field (using its `@id`)
- Normalize that field
- Optionally group by a categorical field

In [ ]:
# --- Setup: pick one record set and numeric field for EDA ---

# 1. Choose a record set with numeric fields
selected_rs_id = None
numeric_field_id = None
group_field_id = None

# Identify a DataFrame with a numeric column
for rs_id, df in dfs.items():
    for col in df.columns:
        # Simple heuristic: check for int or float columns
        if pd.api.types.is_numeric_dtype(df[col]):
            selected_rs_id = rs_id
            numeric_field_id = col
            # Try to pick a grouping variable -- arbitrary: find first object type column
            for gcol in df.columns:
                if gcol != numeric_field_id and df[gcol].dtype == 'object':
                    group_field_id = gcol
                    break
            break
    if selected_rs_id:
        break

if selected_rs_id is None or numeric_field_id is None:
    print("No suitable numeric field found in dataset.")
else:
    print(f"Using Record Set @id: {selected_rs_id}")
    print(f"Numeric Field @id: {numeric_field_id}")
    if group_field_id:
        print(f"Grouping Field @id: {group_field_id}")

    df = dfs[selected_rs_id]

    # 2. Filtering step: keep values > mean (as an example)
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df[[numeric_field_id]].head())

    # 3. Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # 4. Optional grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its relationship (if any) with the group field (if set).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} in Record Set {selected_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=90)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

In this notebook, you learned how to:
- Load metadata and records of a Croissant-formatted dataset with `mlcroissant` using only `@id`s
- Discover record sets and fields, referencing them via `@id`
- Extract records into DataFrames and perform basic exploratory analysis
- Filter, normalize, and visualize a numeric field, optionally grouped by a categorical attribute

**Next steps:** Tailor filtering/grouping to your analytic goals, or modify this workflow to interrogate domain-specific fields of the FAIR^2 dataset by referring to their `@id`s from the Data Overview above.